In [1]:
import sys
import torch
sys.path.insert(0,"../src")

from cnn_project.train import train_model
from cnn_project.model import EuroSATModel
from cnn_project.evaluate import evaluate_model
from cnn_project.data import create_data_loaders
from pathlib import Path
import matplotlib.pyplot as plt
import optuna

from torch.profiler import (
    profile, schedule, ProfilerActivity, record_function
)

/home/h171138/EuroSAT-Model/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device =torch.device("cuda"if torch.cuda.is_available() else "cpu")

torch.manual_seed(42)
profile_model =  EuroSATModel().to(device)
profile_model.train()

optimizer = torch.optim.Adam(profile_model.parameters(), lr=0.00633)
loss_fn= torch.nn.CrossEntropyLoss()

train_loader, val_loader, test_loader = create_data_loaders(batch_size=512, num_workers=12)

batches = iter(train_loader)

In [3]:
activities = [ProfilerActivity.CPU]
if device.type == "cuda":
    activities.append(ProfilerActivity.CUDA)

with profile(
    activities = activities,
    schedule= schedule(wait =5, warmup =5, active =5, repeat =1),
    record_shapes =False,
    profile_memory =False,
    with_stack=False,
) as prof:
    for _ in range(15):
        with record_function("get_batch"):
            images, labels = next (batches)
        with record_function("copy_to_device"):
            images=images.to(device, non_blocking =True)
            labels= labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        with record_function("forward_and_loss"):
            output = profile_model(images)
            loss =loss_fn(output, labels)

        with record_function("backward"):
            loss.backward()

        with record_function("optimizer_step"):
            optimizer.step()

        prof.step()

USDT:2026-10-28 13:11:51 54613:54613 SyncActivityProfilerHandler.cpp:39] profiler_start
USDT:2026-10-28 13:11:52 54613:54613 SyncActivityProfilerHandler.cpp:46] profiler_stop


In [4]:
stats = prof.key_averages()

print("Cpu operations and writing:")
print(stats.table(sort_by = "cpu_time_total",row_limit=15))

if device.type == "cuda":
    print("GPU operations:")
    print(stats.table(sort_by="self_device_time_total", row_limit=15))

Cpu operations and writing:
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                  cudaDeviceSynchronize        86.36%     339.828ms        86.90%     341.934ms     341.934ms       0.000us         0.00%       0.000us       0.000us             1  
                                          ProfilerStep*         0.33%       1.306ms         9.74%      38.328ms       7.666ms       0.000us         0.00%      73.661ms      14.732